# 03 - Exploratory Data Analysis (EDA)

This notebook performs EDA on the transaction and identity data from data/interim.
Data is stored as Parquet chunks and will be loaded with Dask to handle the full dataset without OOM.

Steps:
1. Load transaction and identity chunks with Dask
2. Merge datasets
3. Analyze shape, data types, missingness
4. Inspect target distribution (isFraud)
5. Explore numeric correlations
6. Save a small sample to data/processed for fast iteration


In [ ]:
import os
from pathlib import Path
import glob

import pandas as pd
import numpy as np
import dask.dataframe as dd
import matplotlib.pyplot as plt
import seaborn as sns

# Paths
INTERIM_DIR = Path('/home/laenra/projects/ieee-cis-fraud-detection/data/interim')
PROCESSED_DIR = Path('/home/laenra/projects/ieee-cis-fraud-detection/data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load data with Dask

In [ ]:
# Load transaction chunks
tx_pattern = str(INTERIM_DIR / 'train_transaction_chunks' / 'train_transaction_chunk_*.parquet')
tx_files = sorted(glob.glob(tx_pattern))
print(f'Found {len(tx_files)} transaction chunks')
ddf_tx = dd.read_parquet(tx_files)
print(f'Transactions shape (lazy): {ddf_tx.shape}')
print(f'Transactions columns: {list(ddf_tx.columns)}')
print(f'Transactions dtypes:\n{ddf_tx.dtypes}')

In [ ]:
# Load identity chunks
id_pattern = str(INTERIM_DIR / 'train_identity_chunks' / 'train_identity_chunk_*.parquet')
id_files = sorted(glob.glob(id_pattern))
print(f'Found {len(id_files)} identity chunks')
ddf_id = dd.read_parquet(id_files)
print(f'Identity shape (lazy): {ddf_id.shape}')
print(f'Identity columns: {list(ddf_id.columns)}')
print(f'Identity dtypes:\n{ddf_id.dtypes}')

## 2. Merge datasets

In [ ]:
# Merge on TransactionID
ddf = ddf_tx.merge(ddf_id, on='TransactionID', how='left')
print(f'Merged shape (lazy): {ddf.shape}')
print(f'Total columns: {len(ddf.columns)}')

## 3. Compute and sample for quick analysis

In [ ]:
# Get sample (1%) for fast summaries
SAMPLE_FRAC = 0.01
sample = ddf.sample(frac=SAMPLE_FRAC, random_state=42).compute()
print(f'Sample shape: {sample.shape}')
print(f'\nSample head:')
display(sample.head())

In [ ]:
# Basic info on the full dataset (via Dask)
full_rows = len(ddf)
print(f'Full dataset rows: {full_rows:,}')
print(f'\nData types:\n{ddf.dtypes}')

## 4. Missingness Analysis

In [ ]:
# Compute null counts for full dataset
null_counts = ddf.isnull().sum().compute()
null_pct = 100 * null_counts / full_rows
missing_df = pd.DataFrame({
    'Column': null_counts.index,
    'Null_Count': null_counts.values,
    'Null_Percent': null_pct.values
}).sort_values('Null_Count', ascending=False)
missing_df = missing_df[missing_df['Null_Count'] > 0]
print('Columns with missing values:')
display(missing_df)

## 5. Target Variable Distribution

In [ ]:
# Check isFraud distribution on full data
if 'isFraud' in ddf.columns:
    target_dist = ddf['isFraud'].value_counts().compute().sort_index()
    target_pct = 100 * target_dist / full_rows
    print('Target distribution (isFraud):')
    for val, cnt in target_dist.items():
        pct = 100 * cnt / full_rows
        print(f'  {val}: {cnt:,} ({pct:.2f}%)')
    
    # Plot
    fig, ax = plt.subplots()
    target_dist.plot(kind='bar', ax=ax)
    ax.set_title('Target Distribution (isFraud)')
    ax.set_xlabel('isFraud')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / '03_target_distribution.png', dpi=100)
    plt.show()
else:
    print('isFraud column not found')

## 6. Numeric Features Summary

In [ ]:
# Describe numeric columns on sample (faster)
numeric_cols = sample.select_dtypes(include=['number']).columns.tolist()
print(f'Numeric columns count: {len(numeric_cols)}')
print(f'\nNumeric summary (on sample):')
display(sample[numeric_cols].describe())

## 7. Correlation Analysis

In [ ]:
# Compute correlations on full data (using Dask)
# This may take a moment for many features
print('Computing correlations on full dataset...')
corr_full = ddf[numeric_cols].corr().compute()
print('Correlations computed!')

# Show correlation with target if it's numeric
if 'isFraud' in corr_full.index:
    target_corr = corr_full['isFraud'].sort_values(ascending=False)
    print('\nTop features correlated with isFraud:')
    display(target_corr.head(15))
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    target_corr.head(15).plot(kind='barh', ax=ax)
    ax.set_title('Top 15 Features Correlated with isFraud')
    ax.set_xlabel('Correlation')
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / '03_target_correlations.png', dpi=100)
    plt.show()

## 8. Categorical Features

In [ ]:
# Identify categorical columns
cat_cols = sample.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns: {len(cat_cols)}')
print(f'Columns: {cat_cols}')

# Show cardinality on full data
print('\nCardinality (unique values) for categorical features:')
for col in cat_cols[:10]:  # Show first 10
    nunique = ddf[col].nunique().compute()
    print(f'  {col}: {nunique:,}')

## 9. Save sample for fast iteration

In [ ]:
# Save sample as CSV
sample_path = PROCESSED_DIR / 'eda_sample.csv'
sample.to_csv(sample_path, index=False)
print(f'Saved sample to {sample_path}')
print(f'Sample size: {sample.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

## 10. Data Quality Report

In [ ]:
print('\n=== DATA QUALITY REPORT ===')
print(f'Total Rows: {full_rows:,}')
print(f'Total Columns: {len(ddf.columns)}')
print(f'Numeric Features: {len(numeric_cols)}')
print(f'Categorical Features: {len(cat_cols)}')
print(f'Columns with Missing Values: {len(missing_df)}')
print(f'\nFraud Rate (if target exists):')
if 'isFraud' in ddf.columns:
    fraud_cnt = (ddf['isFraud'] == 1).sum().compute()
    fraud_pct = 100 * fraud_cnt / full_rows
    print(f'  Fraudulent: {fraud_cnt:,} ({fraud_pct:.2f}%)')
    print(f'  Legitimate: {full_rows - fraud_cnt:,} ({100-fraud_pct:.2f}%)')